# 02 — Feature Engineering

This notebook documents every feature extracted from the audio and
investigates which features are most discriminative for the five cry classes.

**Sections**
1. Feature extraction pipeline recap
2. Extract & cache features
3. Feature dimensionality overview
4. Per-class feature statistics
5. Feature importance (mutual information)
6. PCA / t-SNE visualisation
7. Save final feature matrices

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif

from src.config import (
    CLASSES, SAMPLE_RATE, N_MFCC, RANDOM_STATE, TEST_SIZE,
    FEATURES_DIR, PLOTS_DIR,
)
from src.data_loader import load_dataset
from src.feature_extraction import (
    extract_features_batch, save_features,
)

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Feature Extraction Pipeline

| Component | Dimensions | Description |
|-----------|-----------|-------------|
| MFCC summary | 40 coeff x 6 stats = **240** | mean, std, min, max, skew, kurtosis of each MFCC |
| Delta-MFCC summary | 40 x 6 = **240** | First-order temporal derivative |
| Delta2-MFCC summary | 40 x 6 = **240** | Second-order temporal derivative |
| Spectral descriptors | 5 x 6 = **30** | ZCR, centroid, bandwidth, rolloff, RMS |
| **Total** | | **750** |

## 2. Extract & Cache Features

In [ ]:
X_raw, y, file_paths = load_dataset()
print(f'Loaded {len(X_raw)} clips')

In [ ]:
clip_features, mfcc_sequences = extract_features_batch(X_raw, return_frame_level=True)
print(f'Clip feature matrix : {clip_features.shape}')
print(f'MFCC sequences      : {len(mfcc_sequences)} sequences, first shape = {mfcc_sequences[0].shape}')

## 3. Feature Dimensionality Overview

In [ ]:
stat_names = ['mean', 'std', 'min', 'max', 'skew', 'kurtosis']
feature_names = []
for prefix in ['mfcc', 'delta1', 'delta2']:
    for coeff in range(N_MFCC):
        for stat in stat_names:
            feature_names.append(f'{prefix}_{coeff}_{stat}')
for desc in ['zcr', 'centroid', 'bandwidth', 'rolloff', 'rms']:
    for stat in stat_names:
        feature_names.append(f'{desc}_{stat}')

print(f'Expected features: {len(feature_names)}')
print(f'Actual features  : {clip_features.shape[1]}')
assert len(feature_names) == clip_features.shape[1], 'Dimension mismatch!'

In [ ]:
df_feat = pd.DataFrame(clip_features, columns=feature_names)
df_feat['class'] = [CLASSES[label] for label in y]
df_feat.describe().T.head(20)

## 4. Per-Class Feature Statistics

In [ ]:
# Show mean of first 12 MFCC-mean features grouped by class
mfcc_mean_cols = [f'mfcc_{i}_mean' for i in range(12)]
class_means = df_feat.groupby('class')[mfcc_mean_cols].mean()

fig, ax = plt.subplots(figsize=(12, 6))
class_means.T.plot(kind='bar', ax=ax)
ax.set_title('Mean of First 12 MFCC Means — by Class')
ax.set_ylabel('Value')
ax.set_xlabel('MFCC Coefficient')
ax.legend(title='Class', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'mfcc_means_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Importance — Mutual Information

In [ ]:
mi = mutual_info_classif(clip_features, y, random_state=RANDOM_STATE, n_neighbors=5)
mi_series = pd.Series(mi, index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
mi_series.head(30).plot(kind='barh', ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Mutual Information')
ax.set_title('Top 30 Features by Mutual Information with Class Label')
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'feature_importance_mi.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. PCA & t-SNE Visualisation

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(clip_features)

# PCA — explained variance
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(cumvar)+1), cumvar, marker='.', markersize=3)
ax.axhline(0.95, color='r', linestyle='--', label='95% variance')
n95 = np.argmax(cumvar >= 0.95) + 1
ax.axvline(n95, color='g', linestyle='--', label=f'{n95} components')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance')
ax.set_title('PCA — Cumulative Explained Variance')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'pca_explained_variance.png', dpi=150)
plt.show()
print(f'Components for 95% variance: {n95}')

In [ ]:
# 2D PCA scatter
pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for idx, cls in enumerate(CLASSES):
    mask = y == idx
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], label=cls, alpha=0.6, s=20)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA — 2D Projection')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'pca_2d_scatter.png', dpi=150)
plt.show()

In [ ]:
# t-SNE
tsne = TSNE(n_components=2, random_state=RANDOM_STATE, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
for idx, cls in enumerate(CLASSES):
    mask = y == idx
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=cls, alpha=0.6, s=20)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE — 2D Projection')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'tsne_2d_scatter.png', dpi=150)
plt.show()

## 7. Save Final Feature Matrices

In [ ]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

X_train, X_test = clip_features[train_idx], clip_features[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

save_features(X_train, y_train, tag='train')
save_features(X_test, y_test, tag='test')

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

---
## Summary

- **750-dimensional** feature vector per clip (MFCC + deltas + spectral)
- Mutual information ranking identifies the most discriminative coefficients
- PCA shows ~95% variance captured in a reduced subspace
- t-SNE reveals cluster structure across classes
- Features saved as `.npz` for downstream model training